In [1]:
print('ritu')

ritu


In [2]:
import ast
from datetime import datetime
from langchain_ai21.chat_models import ChatAI21
from langchain_groq import ChatGroq
from langchain_core.output_parsers import PydanticOutputParser
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import SystemMessage,BaseMessage,HumanMessage,AIMessage,ToolMessage
from langgraph.types import interrupt,Command 
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from typing import TypedDict,Annotated, List, Dict, Literal,Optional
from langgraph.checkpoint.postgres import PostgresSaver
from psycopg_pool import ConnectionPool
from pydantic import BaseModel
from json_repair import repair_json
import re
from typing import List
from dotenv import load_dotenv
import json
import os
load_dotenv()
DB_URL = os.getenv("PPT_URL")

class PptState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    outline: List
    detailed_slides: List[Dict]
    current_slide_index: int
    feedback: str
    topic: str
    research_data: str
    action: Literal[ "continue_slide", "update_outline", "update_slide", "complete", '' ]
    tool_caller: Literal["generate_outline","generate_slide_detail"]
class DetailedPoint(BaseModel):
    key_point: str
    explanation: str
class DetailedSlideOutput(BaseModel):
    slide_number: int
    slide_title: str
    layout: Literal[
        "bullets",
        "bullets_with_text",
        "paragraph",
        "two_column",
        "mixed"
    ]
    intro_line: Optional[str] = None
    bullet_points: Optional[List[str]] = None
    supporting_text: Optional[str] = None
    paragraphs: Optional[List[str]] = None

model = ChatGroq(model="llama-3.3-70b-versatile")
searchTool = TavilySearchResults(max_results=1)
tools = [searchTool]
model_with_tools = model.bind_tools(tools)
detailed_parser = PydanticOutputParser(pydantic_object= DetailedSlideOutput)


OUTLINE_SYSTEM_PROMPT = SystemMessage(content="""
Generate slide titles.

Return a Python list of strings.

Example:
["Title 1", "Title 2", "Title 3"]

Rules:
- Exact number of titles
- No explanation
- Only list
""")




# DETAIL_SYSTEM_PROMPT = SystemMessage(
# content=f"""
# You are a professional presentation designer.

# Your job is to generate visually balanced slide content for a PowerPoint presentation.

# For each slide choose the most appropriate layout or a mix of layouts  and generate structured content.

# Available layouts:
# - bullets
# - bullets_with_text
# - paragraph
# - mixed


# {detailed_parser.get_format_instructions()}

# CRITICAL REQUIREMENTS:
# - EVERY slide must include AT LEAST ONE statistic or research finding
# - Cite source: "According to [Organization/Study], [specific fact]"
# - Include exact numbers, dates, percentages
# - Use search tool to find: "[topic] + statistics 2025" or "[topic] + latest research"

# General Principles:
# Slides should be informative but not crowded.
# A good slide often combines short explanations with bullet points.

# Content Elements:
# Slides may contain combination of:
# - intro_line (a introductory sentence)
# - bullet_points (key ideas)
# - supporting_text (a short insight or explanation)
# - paragraphs (short explanation text)


# Layout Guidelines:

# bullets
# - 6-8 bullet points
# - each bullet 6–12 words
# - may optionally include a intro_line before bullets
# - Some bullet slides should include a short explanation sentence before or after the bullets.

# bullets_with_text
# - 5-8 bullet points
# - include supporting_text (1 short explanation sentence)
# - may optionally include intro_line
# - Some bullet slides should include a short explanation sentence before or after the bullets.

# paragraph
# - 3-5 short paragraphs
# - each paragraph max 40 words
# - optionally include a short intro_line

# Slides may contain a combination of:

# - intro_line (1 explanation sentence)
# - bullet_points (3–5 bullets)
# - paragraphs (1 short paragraph)
# - supporting_text (1 insight sentence)

# Good slides often combine elements.
# Example structure:

# intro_line
# bullet_points
# supporting_text


# Layout Distribution Rules:

# - 40–50% slides → bullets
# - 20–30% slides → bullets_with_text
# - 10–20% slides → paragraph

# Variation Rules:
# Do NOT generate the same layout repeatedly.
# Use different content styles across slides.


# Content Density Rules:
# Slides should contain roughly 80-120 words total.
# Content must fit comfortably on a PowerPoint slide.

# Quality Rules:
# - Bullet points should be concise and informative.
# - Avoid repeating the same wording.
# - Ensure the slide content is clear when presented visually.
# Return JSON only.
# """
# )

DETAIL_SYSTEM_PROMPT = SystemMessage(content="""
Generate slide content in JSON.

Format:
{
  "slide_number": number,
  "slide_title": "string",
  "layout": "bullets | bullets_with_text | paragraph | mixed",
  "intro_line": "string (optional)",
  "bullet_points": ["string"] (optional),
  "supporting_text": "string (optional)",
  "paragraphs": ["string"] (optional)
}

Rules:
- Include at least 1 statistic with source
- Use clear, short content (80–120 words)
- Bullet points: 5–7 items, short phrases
- Keep slide visually balanced
- No explanation, JSON only
""")




def generate_outline_node(state: PptState):
    """Step 1: Generate presentation outline"""
    messages = state["messages"] + [OUTLINE_SYSTEM_PROMPT]

    result = model_with_tools.invoke(messages)
    output = {
        'messages':[result],
        'current_slide_index':0,
        "tool_caller": "generate_outline",
            } 
    if result.content:
        try:
            # output['outline'] = outline_parser.parse(repair_json(str(result.content))).model_dump()
            output['outline'] = ast.literal_eval(result.content)
            # print("output['outline']",output['outline'])
        except json.JSONDecodeError as e:
            print('generate_outline_node',e)
    return output


def generate_slide_detail_node(state: PptState):
    """Generate detailed content using research data"""
    print('inside generate_slide_detail_node')
    outline = state['outline']
    current_index = state['current_slide_index']
    detailed_slides = state.get('detailed_slides', [])
    total_slides = len(outline)
    # current_slide = outline['slides'][current_index]
    current_slide = outline[current_index]
    # print('current_slide',current_slide)
    
    
    if current_index >= total_slides:
        print('inside end')
        return {"action": "complete"}
    
    # Get research data for this slide
    # research = state.get('research_data', {}).get(current_slide['slide_title'], [])
    research = state.get('research_data', {})
    print('research',research)
    
    output = {
        "tool_caller": "generate_slide_detail",
        "current_slide_index": current_index
    }
    
    if state['action'] == "update_slide":
        # Handle feedback case (existing logic)
        feedback = state['feedback']
        last_slide = detailed_slides.pop()
        prompt = HumanMessage(content=f"""
Update this slide with research data:
Research: {research}
Current Content: {last_slide}
Feedback: {feedback}
        """)
    else:
#         prompt = HumanMessage(content=f"""
# Generate slide using this research data:
# Presentation: {state['topic']}
# Slide: {current_slide} (#{current_slide})
# RESEARCH RESULTS: {research}

# MANDATORY: Include 2-3 specific statistics with sources in bullets/intro/supporting_text
# Format: "X million tons annually (Source {datetime.now().year})"
#         """)

        prompt = HumanMessage(content=f"""
Slide: {current_slide}
Topic: {state['topic']}

Use this research:
{research}

Include 1–2 stats with source.
""")

    # print("DETAIL_SYSTEM_PROMPT",DETAIL_SYSTEM_PROMPT)
    messages = [DETAIL_SYSTEM_PROMPT, prompt]

    # result = model_with_tools.invoke(messages)
    result = model.invoke(messages)
    print('result',result)
    
    if result.content:
        try:
            new_slide = detailed_parser.parse(repair_json(str(result.content))).model_dump()
            detailed_slides.append(new_slide)
            output['detailed_slides'] = detailed_slides
            output['current_slide_index'] = current_index + 1
        except json.JSONDecodeError:
            pass
    
    if output['current_slide_index'] == total_slides:
        output["action"] = "complete"
    output['messages'] = [result]
    print('output',output)
    return output



def research_slide_node(state: PptState):
    print('inside research_slide_node')
    """Research current slide before content generation"""
    slide_title = state['outline'][state['current_slide_index']]
    # slide_title = current_slide
    
    # Use search tool to get real data
    research_query = f"{slide_title} statistics {datetime.now().year} research report"
    # print('research_query',research_query)
    # return
    research_result = searchTool.invoke({"query": research_query})

    # print('research_result',research_result)
    # print('type',type(research_result))
    # print('0',research_result[0])
    # print('type 0',type(research_result[0]))
    # print('content',research_result[0]['content'])
    # return
    return {
        # "research_data": {slide_title: research_result},
        "research_data": research_result[0]['content'],
        "messages": [ToolMessage(content=str(research_result), tool_call_id="research")],
        "tool_caller": "generate_slide_detail"
    }


def route_after_tools(state: PptState):
    return state["tool_caller"]

def human_decision(state: PptState):
    decision = interrupt({})
    if decision['action'] == "update_outline":

        return {
            'action': "update_outline",
            "messages":[decision['feedback']]
            }
    elif decision['action'] == 'continue_slide':
        return {'action':'continue_slide'}
    elif decision['action'] == 'update_slide':
        return {'action':'update_slide'}
def route_after_human(state: PptState):
    action = state['action']
    if action == 'update_outline':
        return "generate_outline"
    elif action in ('continue_slide', 'update_slide'):
        return "generate_slide_detail"
    elif action == 'complete':  
        return END
    return END

def tools_condition(state: PptState):
    """Route to tools if last message has tool calls"""
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return "__end__"

def build_workflow():
    workflow = StateGraph(PptState)
    workflow.add_node("generate_outline", generate_outline_node)
    workflow.add_node("research_slide", research_slide_node)
    workflow.add_node("generate_slide_detail", generate_slide_detail_node)
    workflow.add_node("human_decision", human_decision)
    workflow.add_node("tools", ToolNode(tools))
    
    workflow.add_edge(START, "generate_outline")

    workflow.add_conditional_edges( #done
        "generate_outline",
        tools_condition,
        {
            "tools": "tools",
            "__end__": "human_decision",
        },
    )

    workflow.add_conditional_edges(
        "human_decision",
        route_after_human,
        {
            "generate_outline": "generate_outline",
            "generate_slide_detail": "research_slide",
            END: END,
        },
    )

    workflow.add_conditional_edges(
        "research_slide",
        tools_condition,
        {"tools": "tools", "__end__": "generate_slide_detail"},
    )

    workflow.add_conditional_edges(
        "generate_slide_detail",
        tools_condition,
        {
            "tools": "tools",
            "__end__": "human_decision",
        },
    )
    workflow.add_conditional_edges(
        "tools",
        route_after_tools,
        {
            "generate_outline": "generate_outline",
            "generate_slide_detail": "generate_slide_detail",
        },
    )

    return workflow
def create_ckeckpointer_and_graph(db_url: str):
    if not db_url:
        raise ValueError('Database Url environment variable not set')
    connection_kwargs = {
            "autocommit": True,
            "prepare_threshold": 0,
        }
    pool = ConnectionPool(
        conninfo=db_url,
            max_size=20,
            kwargs=connection_kwargs,
    )
    checkpointer = PostgresSaver(pool)
    checkpointer.setup()
    workflow = build_workflow()
    graph = workflow.compile(checkpointer=checkpointer)
    return checkpointer, graph

print('run')

c:\Users\kaushal\Desktop\me\AI-Powered PPT Generator\pptenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


run


C:\Users\kaushal\AppData\Local\Temp\ipykernel_14248\3044421120.py:54: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  searchTool = TavilySearchResults(max_results=1)


In [3]:
import random

topics = ["Impact of Social Media on Human Relationships",
"Plastic Pollution and Its Effect on the Environment",
"Importance of Mental Health in Modern Life",
"Urbanization and Its Impact on Cities",
"Healthy Lifestyle and the Role of Exercise",
"Electric Vehicles and the Future of Transportation",
"The Importance of Time Management for Students",
"Global Warming: Causes and Solutions",
"Women Empowerment in Modern Society",
"The Role of Education in Personal Development"]


checkpointer, graph = create_ckeckpointer_and_graph(DB_URL)

# topic = "Importance of Mental Health in Modern Life"
# topic = "neta ji shubhash carnd bosh"

topic = random.choice(topics)
num_slides = 5
prompt = HumanMessage(
    content=f"""
Create EXACTLY {num_slides} slide titles for a presentation on:

Topic: {topic}

Do not create fewer or more slides.
"""
)
config = {'configurable':{'thread_id':'22-03-26-1'}}
state = {
            "messages": [prompt],
            "topic":topic,
            "outline": {},
            "detailed_slides": [],
            "current_slide_index": 0,
            "feedback": "",
            "action": "",
            "tool_caller": "generate_outline",
        }

tokens = {
    'completion_tokens':[],
    'prompt_tokens':[],
    "total_tokens":[],
    'completion_tokens_t':0,
    'prompt_tokens_t':0,
    "total_tokens_t":0

}

print('run')

result = graph.invoke(state,config = config)

print('result',result)
result = result['messages'][-1]

tokens['completion_tokens'].append(result.response_metadata['token_usage']['completion_tokens'])
tokens['prompt_tokens'].append(result.response_metadata['token_usage']['prompt_tokens'])
tokens['total_tokens'].append(result.response_metadata['token_usage']['total_tokens'])

tokens['completion_tokens_t'] += result.response_metadata['token_usage']['completion_tokens']
tokens['prompt_tokens_t'] += result.response_metadata['token_usage']['prompt_tokens']
tokens['total_tokens_t'] += result.response_metadata['token_usage']['total_tokens']

run
result {'messages': [HumanMessage(content='\nCreate EXACTLY 5 slide titles for a presentation on:\n\nTopic: Electric Vehicles and the Future of Transportation\n\nDo not create fewer or more slides.\n', additional_kwargs={}, response_metadata={}, id='3a4f5496-1cf2-4ccd-979e-eff344ab1143'), AIMessage(content='["Introduction to Electric Vehicles", "The Benefits of Electric Vehicles", "Challenges Facing Electric Vehicle Adoption", "The Future of Transportation: Electric Vehicle Trends", "Conclusion: Electric Vehicles and a Sustainable Future"]', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 352, 'total_tokens': 394, 'completion_time': 0.150304178, 'completion_tokens_details': None, 'prompt_time': 0.034241771, 'prompt_tokens_details': None, 'queue_time': 0.048601678, 'total_time': 0.184545949}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprob

In [4]:
tokens
# {'completion_tokens': [131],
#  'prompt_tokens': [728],
#  'total_tokens': [859],
#  'completion_tokens_t': 131,
#  'prompt_tokens_t': 728,
#  'total_tokens_t': 859}

# [{'slide_number': 1, 'slide_title': 'Introduction to Neta Ji'},
#   {'slide_number': 2, 'slide_title': 'Early Life of Shubhash Chandra Bosh'},
#   {'slide_number': 3, 'slide_title': 'Indian National Army and Its Impact'},
#   {'slide_number': 4, 'slide_title': 'Contribution to Indian Independence'},
#   {'slide_number': 5,'slide_title': 'Legacy of Neta Ji Shubhash Chandra Bosh'}]


{'completion_tokens': [42],
 'prompt_tokens': [352],
 'total_tokens': [394],
 'completion_tokens_t': 42,
 'prompt_tokens_t': 352,
 'total_tokens_t': 394}

In [5]:
# config = {'configurable':{'thread_id':'15-03-26-3'}}
# for i in range(3):
state = Command(resume={
    "action":'continue_slide'
})
result2 = graph.invoke(state,config = config)
result1 = result2['messages'][-1]
tokens['completion_tokens'].append(result1.response_metadata['token_usage']['completion_tokens'])
tokens['prompt_tokens'].append(result1.response_metadata['token_usage']['prompt_tokens'])
tokens['total_tokens'].append(result1.response_metadata['token_usage']['total_tokens'])

tokens['completion_tokens_t'] += result1.response_metadata['token_usage']['completion_tokens']
tokens['prompt_tokens_t'] += result1.response_metadata['token_usage']['prompt_tokens']
tokens['total_tokens_t'] += result1.response_metadata['token_usage']['total_tokens']
    # print(result1)

inside research_slide_node
inside generate_slide_detail_node
research ## Introduction to Electric Vehicles

The electric vehicle (EV) industry has rapidly transformed the global auto industry, emerging as a cornerstone of sustainable transportation. Electric vehicles, including battery electric vehicles (BEVs) and plug-in hybrids (PHEVs), offer a cleaner, more efficient alternative to traditional internal combustion engine (ICE) vehicles. As environmental concerns and regulatory pressures mount, demand for electric vehicles continues to rise across major markets. [...] Battery electric vehicles account for the majority of new electric vehicle demand. Plug-in hybrids and hybrid vehicles continue to serve as transitional technologies in smaller markets where charging infrastructure is less mature. In many major markets, EV purchases now represent a meaningful share of total vehicle sales. Analysts predict that 2025 will be a pivotal year for EVs, with electrified vehicles potentially com

In [ ]:
result2

In [ ]:
data = [{'slide_number': 1,
   'slide_title': 'Introduction to Mental Health',
   'layout': 'bullets_with_text',
   'intro_line': 'Mental health is a critical aspect of modern life',
   'bullet_points': ['50% of mental illnesses show symptoms by age 14 (Source: 2026 mental health data)',
    '75% of mental illnesses show symptoms by age 24',
    '1 in 3 youth navigate mental, emotional, or behavioral challenges',
    '60% of youth with major depression do not receive treatment',
    '29% of high school students reported poor mental health in the last 30 days'],
   'supporting_text': 'According to 2026 data, nearly 50% of all mental illnesses begin to show symptoms by age 14, highlighting the need for early awareness and intervention (Source: 2026 mental health statistics).',
   'paragraphs': None},
  {'slide_number': 1,
   'slide_title': 'The Impact of Modern Life on Mental Wellbeing',
   'layout': 'bullets_with_text',
   'intro_line': 'Mental health is a central feature of the modern human experience',
   'bullet_points': ['84% of employees face mental health challenges (source: recent workplace trends survey)',
    '57% experience moderate to high burnout',
    "50% of U.S. employees are 'quiet quitting' due to stress",
    'Depression and anxiety cost the global economy $1 trillion per year (source: WHO)',
    "Technology's 'digital friction' impacts psychological health"],
   'supporting_text': 'The digital-first world and workplace demands strain mental wellbeing',
   'paragraphs': None},
  {'slide_number': 1,
   'slide_title': 'Breaking the Stigma Around Mental Illness',
   'layout': 'bullets',
   'intro_line': 'Mental health affects us all',
   'bullet_points': ['26.4% of females experience mental illness (source: mental health research)',
    'Males are 4 times more likely to die by suicide',
    '53.2% of LGBTQ+ community experience mental illness',
    'Only 32.5% of Asian Americans with mental illness receive treatment',
    'Mental illness is a central feature of the modern human experience'],
   'supporting_text': None,
   'paragraphs': None},
  {'slide_number': 1,
   'slide_title': 'Strategies for Maintaining Good Mental Health',
   'layout': 'bullets_with_text',
   'intro_line': "Prioritizing mental health is crucial in today's fast-paced world.",
   'bullet_points': ['Practice self-care',
    'Exercise regularly',
    'Mindfulness and meditation',
    'Connect with nature',
    'Seek social support',
    'Get enough sleep'],
   'supporting_text': 'According to McKinsey reports, creating conditions for a supported nervous system is key to mental clarity and emotional strength. In fact, 75% of people experience improved mental health with regular exercise (Source: World Health Organization). By incorporating these strategies, individuals can lead better, make values-aligned decisions, and regain control over their lives.',
   'paragraphs': None},
  {'slide_number': 10,
   'slide_title': 'Conclusion and Call to Action',
   'layout': 'bullets_with_text',
   'intro_line': 'Prioritizing mental health is crucial in modern life.',
   'bullet_points': ['1 in 4 experience mental health issues (WHO)',
    'Mental health issues cost $2.5 trillion annually (WHO)',
    'Self-care reduces stress',
    'Seek professional help',
    'Support loved ones',
    'Break the stigma'],
   'supporting_text': "Let's take action to promote mental well-being and create a healthier community.",
   'paragraphs': None}]

In [ ]:
from ppt_generator import PPTGenerator

# output_path = os.path.join(MEDIA_DIR,filename)
ppt_obj = PPTGenerator()
ppt_obj.generate_from_list(data)

ppt_obj.save()

In [ ]:
searchTool = TavilySearchResults(max_results=1)
q = {'query': 'what is python'}
searchTool.invoke(q)


this give


"HTTPError('432 Client Error:  for url: https://api.tavily.com/search')"

In [ ]:
Informative,Persuasive,Instructional,Business